<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
</div>

---


*Full Name:* Faraz Doagoye Tehrani

*Student ID:* 402105998

# Retrieval-Augmented Generation (RAG) – From Scratch to Conversation

## Overview

**Retrieval-Augmented Generation (RAG)** is a technique that enhances a language model’s output by first retrieving relevant information from a knowledge base, then conditioning the generation on that retrieved evidence.

Mathematically, given a query $x$, we want to model:

$
p(y \mid x) = \sum_{z \in \text{Top-}k(x)} p_{\text{retrieve}}(z \mid x) \; p_{\text{generate}}(y \mid x, z)
$

where $z$ is a retrieved document chunk, often treated as a hard selection after top‑k retrieval.

### Dataset used in this notebook

Instead of a manually written toy knowledge base, this version uses a small real subset of **SQuAD v1** from Hugging Face. SQuAD provides real Wikipedia-based contexts, questions, and gold answers. To keep the notebook lightweight, we only use a small validation subset.

### In this notebook you will:

1. **From scratch (no LangChain)**:
   - Load a small real QA dataset
   - Chunk real Wikipedia contexts
   - Create dense embeddings with SentenceTransformers
   - Build a FAISS index for fast nearest-neighbour search
   - Implement a retriever and a GPT‑2 generator
   - Assemble a complete, stateless RAG system

2. **With LangChain**:
   - Integrate memory for multi-turn conversations
   - Implement a **history-aware** retriever that rewrites follow-up questions

3. **With an encoder-decoder generator**:
   - Replace GPT‑2 with FLAN‑T5
   - Compare decoder-only RAG and encoder-decoder RAG on the same retrieved contexts

After completing this notebook, you will understand the core components of RAG, the need for memory in conversational systems, and the architectural difference between decoder-only and encoder-decoder generators.

**Dependencies**: Run the cell below to install required packages.

In [1]:
# Install necessary packages (if not already installed)
import sys
!{sys.executable} -m pip install --quiet torch transformers sentence-transformers faiss-cpu datasets langchain-community langchain-huggingface langchain-core
# We use:
# - datasets for loading a small real SQuAD subset
# - sentence-transformers for dense embeddings
# - faiss-cpu for vector search
# - transformers for GPT-2 and FLAN-T5 generators
# - langchain-community / langchain-huggingface for the vectorstore wrapper in the conversational part


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 58.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires 

In [2]:
import os
import json
import torch
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Part 1: Document Loading & Text Chunking

A RAG system needs a collection of documents to retrieve from. In real applications these are often large texts, so we must split them into manageable **chunks**. Each chunk is a unit that can be embedded and later retrieved.

In this notebook, we use a small real subset of **SQuAD v1** instead of a manually written toy text file. Each SQuAD example contains:

- a real Wikipedia paragraph as `context`,
- a `question`,
- one or more gold `answers`.

We use the contexts as the retrieval corpus and the questions/gold answers for testing.

**Why chunk?**
- Language models have a limited context window.
- Smaller chunks allow more precise retrieval; a long document may contain many topics.

**Chunking trade‑offs**:
- Too small → loss of surrounding context.
- Too large → retrieval may bring irrelevant detail, and generation may exceed token limits.

**Implementation**: We’ll implement a simple recursive character‑based splitter with user‑defined chunk size and overlap.

### Questions 1
1. What would happen if we used whole documents without chunking?
    We would have to embed it into a single vector which would cause a single part to be lost in the whole thing. Also it can cause it to be slow.
2. How does the overlap parameter help preserve continuity between chunks?
    If we don't do it, an answer which may span across multiple chunks may be lost in the way as neither of the chunks are good enough on their own and as a whole they provide the answer.

In [3]:
def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 20) -> list[str]:
    if chunk_overlap >= chunk_size:
        raise ValueError("Chunk overlap must be smaller than chunk size.")
        
    chunks = []
    start = 0
    
    if not text:
        return chunks
        
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        start += (chunk_size - chunk_overlap)
        
        if start >= len(text):
            break
            
    return chunks

In [4]:
from datasets import load_dataset

qa_dataset = load_dataset("squad", split="validation[:300]")

print(qa_dataset)
print()

print("Example row:")
print(qa_dataset[0])
print()

unique_contexts = []
for context in qa_dataset["context"]:
    if context not in unique_contexts:
        unique_contexts.append(context)

print(f"Number of QA examples: {len(qa_dataset)}")
print(f"Number of unique contexts: {len(unique_contexts)}")

CHUNK_SIZE = 500
CHUNK_OVERLAP = 250

chunks = []
for context in unique_contexts:
    context_chunks = chunk_text(context, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks.extend(context_chunks)

print(f"Created {len(chunks)} chunks from real SQuAD contexts.\n\n")

for i in range(min(3, len(chunks))):
    print(f"Chunk {i}:\n")
    print(f"{chunks[i]} ...\n\n")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 300
})

Example row:
{'id': '56be4db0acb8001400a502ec', 'title': 'Super_Bowl_50', 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.', 'question': 'Which NFL team represente

## Part 2: Document Embeddings & Vector Index

To retrieve relevant chunks, we need to embed both the documents and the query into a common dense vector space. We use a pre‑trained sentence transformer to obtain **fixed‑size embeddings**.

**Dense retrieval** computes:

$
\text{sim}(q, d) = \cos(\phi(q), \phi(d)) \quad \text{or} \quad \|\phi(q) - \phi(d)\|_2
$

where $\phi$ is the embedding function.

We’ll store the embeddings in a **FAISS** index for fast approximate (or exact) nearest‑neighbour search.

### Questions 2
1. Explain the difference between sparse retrieval (e.g., TF‑IDF, BM25) and dense retrieval.
   
   Sparse retrieval relies on exact keyword matching, representing text as high-dimensional vectors based on word counts. It is fast and precise for specific names or terms, but it fails to recognize synonyms or paraphrasing.

    Dense retrieval maps text into low-dimensional, continuous vector spaces using neural networks. It matches queries based on underlying semantic meaning, allowing it to connect concepts (like "heart doctor" and "cardiologist") even if they share no exact words.

2. Why is L2 distance often used instead of cosine similarity in FAISS `IndexFlatL2`? (Hint: embedding normalisation)

    Mathematical Equivalence: When vector embeddings are unit-normalized (scaled to a length of 1), minimizing the Euclidean ($L_2$) distance yields the exact same retrieval ranking as maximizing cosine similarity.Also, Calculating $L_2$ distance avoids the extra division and square root operations needed for raw cosine similarity, making it significantly faster and cheaper to compute across millions of vectors in IndexFlatL2.   

In [5]:
# TODO: Load `all-MiniLM-L6-v2` with SentenceTransformer.
# Encode all chunks into float32 dense embeddings.
# Expected output is kept below as a reference.
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(chunks, convert_to_numpy=True)
embeddings = embeddings.astype(np.float32)

print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding type: {embeddings.dtype}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (52, 384)
Embedding type: float32


In [6]:
# TODO: Build a FAISS IndexFlatL2 index.
# Add all chunk embeddings to the index.
# Expected output is kept below as a reference.
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

faiss.normalize_L2(embeddings)

index.add(embeddings)

print(f"FAISS index total elements: {index.ntotal}")
print(f"Is index trained: {index.is_trained}")

FAISS index total elements: 52
Is index trained: True


## Part 3: The Retriever

The retriever is responsible for, given a user query, finding the top‑k most relevant chunks.

Formally:

$
\text{Retrieve}(q, k) = \text{argtopk}_{d \in \mathcal{D}} \; \text{sim}(\phi(q), \phi(d))
$

where $\mathcal{D}$ is the set of all chunk embeddings.

### Question 3
How would you modify the retriever to use **maximum inner product search** (MIPS) instead of L2 distance? When would MIPS be preferred?

    To use Maximum Inner Product Search (MIPS) in FAISS, replace IndexFlatL2 with IndexFlatIP. If you normalize your vectors before adding them to IndexFlatIP, the inner product search becomes mathematically equivalent to cosine similarity.
    
    When it is preferred: MIPS is preferred when the length (magnitude) of the vector contains meaningful information, such as term frequency or product popularity. In these cases, normalizing the vectors would wipe out that critical signal, making raw dot product a better measure of relevance.

In [7]:
# TODO: Implement `retrieve(query, index, embed_model, chunks, top_k)`.
# Embed the query, search FAISS, and return top-k (chunk, distance) pairs.
# Expected output is kept below as a reference.
def retrieve(query: str, index, embed_model, chunks: list[str], top_k: int = 3) -> list[tuple[str, float]]:
    query_embedding = embed_model.encode([query], convert_to_numpy=True)
    query_embedding = query_embedding.astype(np.float32)
    
    faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx != -1:
            results.append((chunks[idx], float(dist)))
            
    return results

question = qa_dataset[0]["question"]
gold_answer = qa_dataset[0]["answers"]["text"][0]

retrieved_results = retrieve(
    query=question, 
    index=index, 
    embed_model=embedding_model, 
    chunks=chunks, 
    top_k=3
)

print(f"Question: {question}")
print(f"Gold answer: {gold_answer}\n")
print("Top retrieved chunks:\n")

for chunk, distance in retrieved_results:
    print(f"Distance {distance:.4f}:\n{chunk}")

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Top retrieved chunks:

Distance 0.6448:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv
Distance 0.6940:
hey joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made eight appearances in the Super Bowl.
Distance 0.8009:
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated the Arizona Cardinals 49–15 in the NFC Champion

## Part 4: The Generator (GPT‑2)

We use a pre‑trained auto‑regressive language model (GPT‑2) to generate an answer conditioned on the retrieved context and the question.

**Input format** we will use:

Context:

chunk 1

chunk 2
...

Question: {user query}

Answer:

GPT‑2 will then continue the string.

### Questions 4
1. Why do we concatenate the context and the question?

        Autoregressive models like GPT-2 generate text by predicting the next token based on all preceding tokens. Concatenating the context before the question forces the model's self-attention mechanism to weight the facts in the text while formulating its response.
   
2. What are the limitations of using a base GPT‑2 model for knowledge‑intensive generation?

        It is prone to hallucination, weak instruction following and has a small context window.
        

In [8]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

gpt2_model_name = "gpt2-medium"  # You can also try "gpt2-medium" if resources allow.
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_model_name)
model = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
model.to(device)
model.eval()

# GPT-2 has no pad token by default; set it to eos for generation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [77]:
# TODO: Implement `build_rag_prompt` and `generate_answer` for GPT-2.
# Concatenate retrieved context and question, then generate an answer.
# Expected output is kept below as a reference.
_FEW_SHOT_EXAMPLES = ( 
    "Context:\n"
    "The Eiffel Tower is located in Paris, France. It was completed in 1889 "
    "and is one of the most visited monuments in the world.\n\n"
    "Question: Where is the Eiffel Tower located?\n\n"
    "Answer: Paris, France. \n(Note: The tower was built as the entrance arch to the 1889 World's Fair.)\n\n"
    "Context:\n"
    "The Great Wall of China was built primarily during the Ming dynasty to "
    "protect against invasions. It stretches over 13,000 miles.\n\n"
    "Question: How long is the Great Wall of China?\n\n"
    "Answer: Over 13,000 miles. \n(Note: It is one of the most extensive architectural feats ever constructed.)\n\n"
)

def build_rag_prompt(retrieved_results: list[tuple[str, float]], question: str) -> str:
    context_text = "\n\n".join([chunk for chunk, _ in retrieved_results])
    return (
        f"{_FEW_SHOT_EXAMPLES}"
        f"\nContext:\n{context_text}\n\nQuestion: {question}\n\nAnswer:"
    )

def generate_answer(prompt: str, model, tokenizer, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,         
            repetition_penalty=1,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output_ids[0][input_length:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    for stop in ["\nContext", "\nQuestion", "\nChat History"]:
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

question = qa_dataset[0]["question"]
gold_answer = qa_dataset[0]["answers"]["text"][0]

retrieved_results = retrieve(
    query=question, 
    index=index, 
    embed_model=embedding_model, 
    chunks=chunks, 
    top_k=1
)

prompt = build_rag_prompt(retrieved_results, question)

gpt2_answer = generate_answer(
    prompt=prompt, 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=50
)
print(f"Question: {question}\n")
print(f"Gold answer: {gold_answer}\n")
print(f"GPT-2 answer: {gpt2_answer}\n")

Question: Which NFL team represented the AFC at Super Bowl 50?

Gold answer: Denver Broncos

GPT-2 answer: The Denver Broncos. 

(Note: The Broncos won the Super Bowl in their first season in the NFL.)



## Part 5: Putting It All Together – Stateless RAG

Now we combine the retriever and the generator into a single pipeline.

Given a question:
1. Retrieve top‑k chunks.
2. Concatenate them into a context string.
3. Feed context + question to the GPT‑2 generator.

Let’s implement a `RAGSystem` class that encapsulates this process.

### Question 5
This system answers each question independently. What problem arises when a user asks a follow‑up question like *“Tell me more about that”* or *“Multiply the previous answer by 2”*?

      When a user asks a follow-up question containing deictic expressions or pronouns (e.g., "that", "the previous answer"), a stateless system cannot resolve what those words refer to. As a result, the embedding model will generate an inaccurate query vector, leading to irrelevant document retrieval and a broken response from the LLM.

In [31]:
# TODO: Implement the stateless `RAGSystem` class.
# It should retrieve chunks, build the context, and generate an answer.
# Expected output is kept below as a reference.
class RAGSystem:
    def __init__(self, index, embed_model, chunks: list[str], generator_model, generator_tokenizer):
        self.index = index
        self.embed_model = embed_model
        self.chunks = chunks
        self.model = generator_model
        self.tokenizer = generator_tokenizer

    def query(self, question: str, top_k: int = 1, max_new_tokens: int = 50) -> str:
        retrieved_results = retrieve(
            query=question, 
            index=self.index, 
            embed_model=self.embed_model, 
            chunks=self.chunks, 
            top_k=top_k
        )
        
        prompt = build_rag_prompt(retrieved_results, question)
        
        answer = generate_answer(
            prompt=prompt, 
            model=self.model, 
            tokenizer=self.tokenizer, 
            max_new_tokens=max_new_tokens
        )
        
        return {
                'answer': answer,
                'retrieved': retrieved_results,
                'prompt' : prompt}

rag = RAGSystem(
    index=index,
    embed_model=embedding_model,
    chunks=chunks,
    generator_model=model,
    generator_tokenizer=tokenizer
)

question = qa_dataset[0]["question"]
gold_answer = qa_dataset[0]["answers"]["text"][0]

rag_answer = rag.query(question=question, top_k=1, max_new_tokens=50)

print(f"Question: {question}")
print(f"Gold answer: {gold_answer}\n")
print("Retrieved chunks:\n")

for chunk, _ in rag_answer['retrieved']:
    print(f"- {chunk}")
    
print(f"\nGPT-2 RAG answer:\n {rag_answer['answer']}")

Question: Which NFL team represented the AFC at Super Bowl 50?
Gold answer: Denver Broncos

Retrieved chunks:

- Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv

GPT-2 RAG answer:
 The Denver Broncos. 

(Note: The Broncos won the Super Bowl in their first season in the NFL.)


## Part 6: The Memory Problem

Let’s simulate a conversation where the second question depends on the first one.

**Stateless RAG will fail or become unstable** because each call is independent. The retriever only sees the latest question, so a follow-up such as *“Where was it played?”* or *“Which team represented the NFC in that game?”* is ambiguous unless the system remembers that the previous topic was **Super Bowl 50**.


In [29]:
q1 = qa_dataset[0]["question"]
ans1 = rag.query(q1, top_k=1, max_new_tokens=80)
print(f"Q: {q1}\nA: {ans1['answer']}\n")

# Follow-up that refers to the previous topic.
# Without memory, the phrase "that game" is ambiguous for the retriever.
q2 = "Which team represented the NFC in that game?"
ans2 = rag.query(q2, top_k=1, max_new_tokens=80)
print(f"Q: {q2}\nA: {ans2['answer']}\n")

# Notice: the second answer may be weaker because stateless RAG does not explicitly
# remember that the previous question was about Super Bowl 50.

Q: Which NFL team represented the AFC at Super Bowl 50?
A: The New England Patriots.

The Patriots were the only team to make the playoffs in the AFC. The team that made the playoffs was the Pittsburgh Steelers, who defeated the New York Jets, New England Patriots, and Indianapolis Colts in the AFC Championship Game.

The Patriots were the only team to make the playoffs in the AFC. The team that made the playoffs was the Pittsburgh Steelers, who

Q: Which team represented the NFC in that game?
A: The Panthers.



In [271]:
retrieved = retrieve(q1, index, embedding_model, chunks, top_k=3)
for chunk, dist in retrieved:
    print(f"dist={dist:.3f}  {chunk[:150]}...\n")

dist=0.563  feating them 20–18 in the AFC Championship Game. They joined the Patriots, Dallas Cowboys, and Pittsburgh Steelers as one of four teams that have made...

dist=0.645  Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football...

dist=0.801  The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). They defeated th...



## Part 7: Adding Memory with LangChain

To enable multi-turn conversations, we need to **maintain a chat history** and use it to reformulate the current question into a standalone query.

In many LangChain tutorials, this is done with functions such as `create_history_aware_retriever` and `create_retrieval_chain`. However, these imports can break across different LangChain versions. To keep this notebook stable, we implement the same logic explicitly while still using LangChain for the FAISS vectorstore wrapper.

We will:

1. Wrap the same FAISS index and embedding model with LangChain’s `FAISS` vectorstore.
2. Store the conversation in a simple `chat_history` list.
3. Rewrite ambiguous follow-up questions into standalone questions.
4. Retrieve documents using the standalone question.
5. Generate the final answer using **FLAN-T5**, which is much better for instruction following than base GPT-2.

### Questions 6
1. Explain how a history-aware retriever works internally. What prompt or rewriting step does it use?

       A history-aware retriever uses a language model to blend the active conversation log with the latest user query before executing a database search. Its rewriting prompt forces the model to swap out ambiguous pronouns (like "it", "there", or "that game") with the concrete entities mentioned in earlier turns, creating a crisp, self-contained question.
   
2. Why can’t we simply pass the whole chat history to the retriever instead of generating a standalone question?

       Feeding a raw multi-turn transcript straight to a retriever floods the vector embedding with conversational noise, filler words, and dead topics. This dilutes the mathematical focus of the embedding vector, causing the system to fetch irrelevant documents instead of targeting your exact, current question.

In [12]:
# TODO: Create LangChain `Document` objects from chunks.
# Then build a FAISS vectorstore using HuggingFace embeddings.
# Expected output is kept below as a reference.
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

documents = [Document(page_content=chunk) for chunk in chunks]

embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(documents, embeddings_model)

print(f"LangChain FAISS index total elements: {vectorstore.index.ntotal}")

/tmp/ipykernel_58/1953968656.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LangChain FAISS index total elements: 52


In [47]:
# TODO: Load `google/flan-t5-small` as an encoder-decoder model.
# Implement a small helper function to generate text with FLAN-T5.
# Expected output is kept below as a reference.
from transformers import T5ForConditionalGeneration, AutoTokenizer

flan_model_name = "google/flan-t5-small"
flan_tokenizer = AutoTokenizer.from_pretrained(flan_model_name)
flan_model = T5ForConditionalGeneration.from_pretrained(flan_model_name)
flan_model.to(device)
flan_model.eval()

def generate_flan_answer(prompt: str, max_new_tokens: int = 50) -> str:
    inputs = flan_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(flan_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            length_penalty=0.5
        )
        
    return flan_tokenizer.decode(output_ids[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [118]:
# TODO: Implement helper functions for conversational RAG.
# Convert chat history to text and rewrite short follow-up questions.
from langchain_core.messages import HumanMessage, AIMessage

def format_chat_history(chat_history: list[tuple[str, str]]) -> str:
    formatted = ""
    for msg in chat_history:
        if isinstance(msg, HumanMessage):
            formatted += f"Human: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            formatted += f"AI: {msg.content}\n"
        elif isinstance(msg, tuple) and len(msg) == 2:
            label = "Human" if msg[0] == "human" else "AI"
            formatted += f"{label}: {msg[1]}\n"
            
    return formatted.strip()

def rewrite_question(chat_history_str: str, current_question: str) -> str:
    if not chat_history_str:
        return current_question
        
    rewrite_prompt = (        
        "Chat History:\n"
        "Human: What is the Eiffel Tower?\n"
        "AI: A famous iron monument in Paris.\n"
        "Follow-up Question: Where was it built?\n"
        "Standalone Question: Where was the Eiffel Tower built?\n\n"
        
        "Chat History:\n"
        "Human: Which tennis player won the men's singles title at the French Open?\n"
        "AI: Rafael Nadal.\n"
        "Follow-up Question: Who won the women's singles title in that tournament?\n"
        "Standalone Question: Who won the women's singles title at the French Open?\n\n"

        "Chat History:\n"
        "Human: What operating system does Apple design for its phones?\n"
        "AI: Apple designs iOS.\n"
        "Follow-up Question: What operating system does Samsung use for its devices?\n"
        "Standalone Question: What operating system does Samsung use for its smartphone devices?\n\n"
                
        f"Chat History:\n"
        f"{chat_history_str}\n"
        f"Follow-up Question: {current_question}\n"
        f"Standalone Question:"
    )
    #I thought since the instructions said that use the flan model at the end it only meant at the end 
    #and not for the standalone question generation.
    return generate_answer(prompt=rewrite_prompt, model=model, tokenizer=tokenizer,max_new_tokens=40)

In [119]:
# TODO: Implement `SimpleConversationalRAG`.
# Steps: rewrite follow-up question, retrieve documents, generate answer.
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

class SimpleConversationalRAG:
    def __init__(self, vectorstore, max_new_tokens: int = 50):
        self.vectorstore = vectorstore
        self.max_new_tokens = max_new_tokens
        self.chat_history = []

    def query(self, question: str, top_k: int = 1) -> dict:
        history_str = format_chat_history(self.chat_history)
        
        standalone_question = rewrite_question(history_str, question)
        
        retrieved_docs = self.vectorstore.similarity_search(standalone_question, k=top_k)
        context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
        
        qa_prompt = (
            f"Answer the question based only on the provided context.\n\n"
            f"Question: {question}\n\n"
            f"Context:\n{context_text}\n\n"
            f"Answer:"
        )
        
        answer = generate_flan_answer(qa_prompt, max_new_tokens=self.max_new_tokens)
        
        self.chat_history.append((question, answer))
        
        return {"answer": answer}

    def clear_memory(self):
        self.chat_history = []

def run_conversational_rag(inputs: dict) -> dict:
    user_input = inputs["input"]
    history_list = inputs.get("chat_history", [])
    
    formatted_history = format_chat_history(history_list)
    
    standalone_q = rewrite_question(formatted_history, user_input)
    
    retrieved_docs = vectorstore.similarity_search(standalone_q, k=1)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    qa_prompt = (
        f"Answer the question based only on the provided context.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question:{standalone_q}\n\n"
        f"Answer:"
    )
    answer_text = generate_flan_answer(qa_prompt, max_new_tokens=80)
    
    return {
        "standalone_question": standalone_q,
        "answer": answer_text,
        "retrieved_context": context_text
    }

rag_chain_with_memory = RunnableLambda(run_conversational_rag)

In [120]:
# Test the corrected memory-based RAG pipeline.
chat_history = []

question_1 = "What was Super Bowl 50?"
response_1 = rag_chain_with_memory.invoke({
    "input": question_1,
    "chat_history": chat_history
})

print("Question 1:", question_1)
print("Standalone question 1:", response_1["standalone_question"])
print("Answer 1:", response_1["answer"])

chat_history.append(("human", question_1))
chat_history.append(("ai", response_1["answer"]))

question_2 = "Where was it played?"
response_2 = rag_chain_with_memory.invoke({
    "input": question_2,
    "chat_history": chat_history
})

print("\nQuestion 2:", question_2)
print("Standalone question 2:", response_2["standalone_question"])
print("Answer 2:", response_2["answer"])
print("\nRetrieved context for turn 2:")
print(response_2["retrieved_context"][:1000], "...")


Question 1: What was Super Bowl 50?
Standalone question 1: What was Super Bowl 50?
Answer 1: an American football game

Question 2: Where was it played?
Standalone question 2: Where was Super Bowl 50 played?
Answer 2: San Francisco Bay Area

Retrieved context for turn 2:
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniv ...


In [121]:
chat_history = []


def ask_question(query: str):
    response = rag_chain_with_memory.invoke({
        "input": query,
        "chat_history": chat_history
    })

    # Update chat history after each turn.
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=response["answer"]))
    return response


# First turn: a real SQuAD-style question.
q1 = qa_dataset[0]["question"]
response1 = ask_question(q1)
print("User:", q1)
print("Standalone:", response1["standalone_question"])
print("Assistant:", response1["answer"])

# Follow-up question that depends on the previous topic.
q2 = "Which team represented the NFC in that game?"
response2 = ask_question(q2)
print("\nUser:", q2)
print("Standalone:", response2["standalone_question"])
print("Assistant:", response2["answer"])


User: Which NFL team represented the AFC at Super Bowl 50?
Standalone: Which NFL team represented the AFC at Super Bowl 50?
Assistant: Denver Broncos

User: Which team represented the NFC in that game?
Standalone: Which NFL team represented the NFC in that game?
Assistant: New Orleans Saints


### Explanation for Question 6

A history-aware retriever first checks whether the latest user question is self-contained. If the question contains references such as *it*, *that game*, or *the previous answer*, the system uses the chat history to rewrite it into a standalone query. For example:

```text
Chat history: What was Super Bowl 50?
Follow-up: Where was it played?
Standalone query: Where was Super Bowl 50 played?
```

The retriever should usually receive this standalone query rather than the whole chat history. Passing the entire chat history directly to the retriever can introduce irrelevant words from previous turns and make vector search less focused. A short standalone query preserves the missing context while keeping retrieval precise.

In this notebook, we implement the history-aware logic explicitly instead of relying on `langchain.chains`, because those imports are version-sensitive. The idea is the same: use memory to rewrite the query, retrieve relevant documents, then answer from the retrieved context.


## Part 8: RAG with an Encoder-Decoder Generator

So far, the retriever has been encoder-based, but the generator has been GPT‑2, which is a decoder-only language model.

In this part, we replace GPT‑2 with an encoder-decoder model, **google/flan-t5-small**. The retrieved context and the user question are passed to the encoder, and the decoder generates the answer.

### Question 7
Implement a RAG pipeline that uses the same retriever as before, but replaces GPT‑2 with an encoder-decoder model such as `google/flan-t5-small`.

1. Compare the GPT‑2-based RAG and the T5-based RAG on at least two questions.

   
        For the first question, T5 successfully extracts the exact correct answer (3), while GPT-2 generates a self-contradictory hallucination (five, including two). However, on the second question, both models fail and incorrectly output Von Miller because his name and stats heavily dominate the beginning of the retrieved context.
           
2. Explain how the retrieved context is used differently in GPT‑2 and T5.

        GPT-2 treats the context as a literal text extension and reads it sequentially from left to right, which causes it to stumble over numbers or blindly parrot phrases it just read. T5 builds a global semantic map of the context first, allowing it to look up specific entities more cleanly, though it can still be distracted if a wrong name appears prominently near the keywords.
3. Why is T5 considered an encoder-decoder model, while GPT‑2 is decoder-only?

       T5 splits the work by using an encoder network to fully digest the context and question before passing those clues to a separate decoder network that writes the final answer. GPT-2 skips the separate digestion phase entirely, using a single network that processes the input prompt and predicts the output text in one continuous token-by-token stream.

In [18]:
# TODO: Implement the T5-based RAG pipeline.
# Use the same retriever, but replace GPT-2 with FLAN-T5.
# Expected output is kept below as a reference.
def t5_rag_pipeline(question: str, top_k: int = 3) -> str:
    retrieved_docs = vectorstore.similarity_search(question, k=top_k)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    qa_prompt = (
        f"Answer the question based only on the provided context.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    
    answer = generate_flan_answer(qa_prompt, max_new_tokens=50)
    return answer, context_text

In [129]:
# TODO: Compare GPT-2 RAG and T5 RAG on at least two questions.
# Use the same retrieved context for both models for a fair comparison.
# Expected output is kept below as a reference.
eval_questions = [
    {"question": qa_dataset[-1]["question"], "gold_answer": qa_dataset[-1]["answers"]["text"][0]},
    {"question": qa_dataset[-2]["question"], "gold_answer": qa_dataset[-2]["answers"]["text"][0]}
]

for i, item in enumerate(eval_questions):
    question = item["question"]
    gold_answer = item["gold_answer"]
    
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    qa_prompt = (
        f"Answer the question based only on the provided context.\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    
    gpt2_answer = generate_answer(prompt=qa_prompt,model=model, tokenizer=tokenizer, max_new_tokens=50)
    t5_answer = generate_flan_answer(qa_prompt, max_new_tokens=50)
    
    print(f"Question: {question}")
    print(f"\nGold answer: \"{gold_answer}\"")
    print(f"\nGPT-2 RAG Answer:{gpt2_answer}")
    print(f"\nT5 Encoder-Decoder RAG Answer:{t5_answer}")
    print(f"\nRetrieved Context:{context_text}")
    
    if i < len(eval_questions) - 1:
        print("=" * 100)

Question: How many interceptions did Aqib Talib have for the regular season?

Gold answer: "three."

GPT-2 RAG Answer:Talib had five, including two interceptions.

T5 Encoder-Decoder RAG Answer:3

Retrieved Context:kles with 109, while Danny Trevathan ranked second with 102. Cornerbacks Aqib Talib (three interceptions) and Chris Harris, Jr. (two interceptions) were the other two Pro Bowl selections from the defense.

720 yards, five touchdowns, 24 receptions, and a 4.7 yards per carry average. Overall, the offense ranked 19th in scoring with 355 points and did not have any Pro Bowl selections.

ith 989 rushing yards and six touchdowns in 13 games, along with Pro Bowl fullback Mike Tolbert, who rushed for 256 yards and caught 18 passes for another 154 yards. Carolina's offensive line also featured two Pro Bowl selections: center Ryan Kalil and guard Trai Turner.
Question: Which linebacker led the Broncos in tackles?

Gold answer: "Brandon Marshall"

GPT-2 RAG Answer:Von Miller

The Bron

### Explanation for Question 7

In both systems, the retriever is exactly the same. The question is embedded using the SentenceTransformer model, and FAISS retrieves the top-k most relevant chunks from the real SQuAD context corpus.

The main difference is the generator architecture.

In the **GPT‑2-based RAG system**, the retrieved context, the question, and the `Answer:` prefix are concatenated into one prompt. GPT‑2 is a **decoder-only** model, so it generates the answer by continuing this prompt from left to right. During generation, each new token can attend only to previous tokens, including the retrieved context and the question.

In the **T5-based RAG system**, the retrieved context and the question are passed to the **encoder**. The encoder reads the full input sequence and produces contextual representations. Then the **decoder** generates the answer autoregressively while attending to the encoder outputs through cross-attention. Therefore, the retrieved context is used as an encoded source sequence rather than only as a text prefix.

T5 is considered an **encoder-decoder** model because it has two separate Transformer components: an encoder for processing the input text and a decoder for generating the output text. GPT‑2 is **decoder-only** because it only contains the autoregressive Transformer decoder stack and generates text by predicting the next token from the previous tokens.

In practice, FLAN‑T5 usually gives cleaner answers for this task than base GPT‑2 because FLAN‑T5 is instruction-tuned and better aligned with question answering. Base GPT‑2 is mainly trained for next-token prediction, so it may continue the prompt instead of directly answering the question.

## Summary

You have:
- Built a **stateless RAG system** from real SQuAD contexts, embeddings, FAISS, and GPT‑2.
- Observed its weakness on follow-up questions.
- Used a LangChain FAISS vectorstore and an explicit **history-aware query rewriting** step to add conversational memory.
- Replaced the unstable GPT‑2 memory generator with **FLAN‑T5** for reformulation and answering.
- Replaced the decoder-only GPT‑2 generator with an **encoder-decoder FLAN‑T5** generator.
- Compared GPT‑2-based RAG and T5-based RAG on the same retrieved contexts.

### Final Theoretical Questions
1. Compare the stateless and conversational RAG pipelines. What are the main differences in the retrieval step?

       Stateless RAG retrieves documents using only the current standalone user query. Conversational RAG alters this step by factoring in the prior chat history. We use an LLM to rewrite the question into a standalone question.
2. In the conversational version, where is the ‘memory’ actually stored?

        It is in an external data structure, such as a Python list. The system puts it in the prompt and the nn is unchanged.
3. How could you extend this system to handle **multiple users**? What would you need to change?
   
        It needs a userid and chatid so it can retrieve the history based on that.
4. Suppose you wanted to replace GPT‑2 with a model that does not support the “reformulate a standalone question” task well. What alternatives could you explore?

       We could try concatonating the previous questions to the current one and pass it to the generating model.
5. Why can an encoder-decoder model such as T5 be more suitable for question answering than a base decoder-only model such as GPT‑2?

        since T5 has a bidirectional encoder, it can access the context and history well whereas the decoder only model goes from left to right and tends to forget the previous ones.

### References
- [SQuAD: 100,000+ Questions for Machine Comprehension of Text](https://rajpurkar.github.io/SQuAD-explorer/)
- [Lewis et al., Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (2020)](https://arxiv.org/abs/2005.11401)
- [FAISS library](https://github.com/facebookresearch/faiss)
- [FLAN-T5 model family](https://huggingface.co/google/flan-t5-small)
